# Structured Prompt PDF RAG with Groq

This notebook matches `rag1_simple_pdf_structured.py`.
It teaches how prompt structure changes output structure.

In [ ]:
!pip -q install openai python-dotenv numpy pypdf tiktoken fastembed

In [ ]:
import os
from getpass import getpass

import numpy as np
import tiktoken
from fastembed import TextEmbedding
from openai import OpenAI
from pypdf import PdfReader

In [ ]:
try:
    from google.colab import files
    uploaded = files.upload()
    pdf_file_path = next(iter(uploaded.keys())) if uploaded else "sample_ocr.pdf"
except ImportError:
    pdf_file_path = input("PDF path (default sample_ocr.pdf): ").strip() or "sample_ocr.pdf"

groq_api_key = os.getenv("GROQ_API_KEY") or getpass("Enter GROQ_API_KEY: ").strip()
chat_model = os.getenv("GROQ_CHAT_MODEL") or "qwen/qwen3.8-27b"
user_question = input("Question: ").strip() or "Summarize this invoice in a structured way for a hospital procurement team."

client = OpenAI(base_url="https://api.groq.com/openai/v1", api_key=groq_api_key)
embedder = TextEmbedding(model_name="BAAI/bge-small-en-v1.5")

In [ ]:
def extract_text_from_pdf(pdf_path: str) -> str:
    reader = PdfReader(pdf_path)
    pages = []
    for page in reader.pages:
        text = page.extract_text()
        if text:
            pages.append(text.strip())
    return "\n".join(pages)

def chunk_text(text: str, chunk_size: int = 500, chunk_overlap: int = 50) -> list[str]:
    tokenizer = tiktoken.get_encoding("cl100k_base")
    tokens = tokenizer.encode(text)
    step = chunk_size - chunk_overlap
    chunks = []
    for start in range(0, len(tokens), step):
        chunks.append(tokenizer.decode(tokens[start:start + chunk_size]))
    return chunks

def get_embedding(text: str) -> list[float]:
    return list(next(embedder.embed([text])))

def cosine_similarity(v1, v2):
    return float(np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2)))

def retrieve_best_chunk(chunks: list[str], question: str) -> str:
    chunk_embeddings = [get_embedding(chunk) for chunk in chunks]
    question_embedding = get_embedding(question)
    similarities = [cosine_similarity(question_embedding, chunk_embedding) for chunk_embedding in chunk_embeddings]
    return chunks[int(np.argmax(similarities))]

def ask_groq(retrieved_context: str, question: str) -> str:
    completion = client.chat.completions.create(
        model=chat_model,
        messages=[
            {
                "role": "system",
                "content": "You are a careful procurement assistant. Use only the provided context. Return the answer with these headings exactly: 1. Document Type 2. Seller 3. Buyer 4. Items and Prices 5. Total Amount 6. Payment and Shipping Terms 7. Risks or Data to Double-Check\n\nContext:\n" + retrieved_context,
            },
            {"role": "user", "content": question},
        ],
    )
    return completion.choices[0].message.content or ""

In [ ]:
raw_pdf_text = extract_text_from_pdf(pdf_file_path)
chunks = chunk_text(raw_pdf_text)
retrieved_context = retrieve_best_chunk(chunks, user_question)
answer = ask_groq(retrieved_context, user_question)
print(answer)